# FOL Diagnostic: Sinh FOL + Z3 Parse Check

**Mục tiêu**: Chạy model FOL fine-tuned trên tập train, xuất CSV, rồi dùng Z3 parser kiểm tra từng formula để tìm:
1. FOL nào sinh sai cú pháp (Z3 parse fail)
2. FOL nào khác với gold label
3. Pattern lỗi phổ biến nhất

**Pipeline**: Load model → Sinh FOL (300 mẫu) → Xuất CSV → Z3 check → Báo cáo

# Báo cáo hiệu suất FOL

https://claude.ai/public/artifacts/47beb715-73a1-497a-962c-04b7fa3ad23f

## 1. Setup

In [1]:
# logic_env (kernel_name) da co san cac goi pin trong requirements.txt.
# Ghim version de KHONG cai de transformers ve ban cu (gay loi TokenizersBackend).
# Bo 'torch' khoi danh sach: tranh pip cai de ban torch lam hong CUDA build.
!pip install -q "transformers==5.5.0" "tokenizers==0.22.2" accelerate bitsandbytes z3-solver


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, os, json, re, ast, time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# === Detect PROJECT_ROOT tự động ===
# Hoạt động trên cả local (Windows) và Modal (Linux /root/...)
# Tìm thư mục chứa "src/data/prompts.py" bằng cách thử các vị trí phổ biến
_cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None
for _candidate in [
    _cwd,                                                    # nếu đang ở project root
    _cwd.parent,                                             # nếu đang ở notebooks/
    _cwd / "Logic_Based_Educational_Queries_Project",        # nếu đang ở repo root
    Path("/root/Logic_Based_Educational_Queries_Project"),   # Modal default
    Path("/root/Exact_2026_Laplace-s_Red_Devils/Logic_Based_Educational_Queries_Project"),
]:
    if (_candidate / "src" / "data" / "prompts.py").exists():
        PROJECT_ROOT = _candidate
        break

# Fallback: tìm bằng glob trong /root
if PROJECT_ROOT is None:
    for p in Path("/root").rglob("src/data/prompts.py"):
        PROJECT_ROOT = p.parent.parent.parent
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy project root (cần chứa src/data/prompts.py). "
        f"CWD = {_cwd}. Hãy cd vào thư mục project trước khi chạy."
    )

# Add src to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Config
MODEL_ID = "Laplaces-Red-Devils/fol-v06-cot-augmented-fol-pretrain-malls-qwen3.5-4"
MAX_SAMPLES = 600
MAX_NEW_TOKENS = 650
RANDOM_SEED = 42
OUTPUT_CSV = PROJECT_ROOT / "notebooks" / "output" / "fol_diagnostic.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"src in path:  {str(PROJECT_ROOT / 'src') in sys.path}")
print(f"Model: {MODEL_ID}")
print(f"Max samples: {MAX_SAMPLES}")
print(f"Output CSV: {OUTPUT_CSV}")

Project root: /root/Logic_Based_Educational_Queries_Project
src in path:  True
Model: Laplaces-Red-Devils/fol-v06-cot-augmented-fol-pretrain-malls-qwen3.5-4
Max samples: 600
Output CSV: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic.csv


## 2. Load Model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
model.eval()
print(f"Model loaded: {MODEL_ID}")

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.41G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Model loaded: Laplaces-Red-Devils/fol-v06-cot-augmented-fol-pretrain-malls-qwen3.5-4


## 3. Load Prompt + Data

In [4]:
from data.prompts import SYSTEM_PROMPT_FOL_SFT, USER_TEMPLATE_FOL_SFT, format_nl_block_numbered

# Đọc max_seq_length từ config
import yaml as _yaml
_cfg_path = PROJECT_ROOT / "configs" / "fol_model.yaml"
_MODEL_MAX_SEQ = 3500
if _cfg_path.is_file():
    _cfg = _yaml.safe_load(_cfg_path.read_text(encoding="utf-8")) or {}
    _MODEL_MAX_SEQ = int((_cfg.get("model") or {}).get("max_seq_length", 3072))

# Load train/dev/test data
SPLITS = ["train", "dev", "test"]
split_dfs = {}
split_unique_dfs = {}
for split in SPLITS:
    split_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / f"{split}.csv")
    split_dfs[split] = split_df
    split_unique_dfs[split] = split_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)
    print(f"{split.title()} data: {len(split_df)} rows, unique records={len(split_unique_dfs[split])}")

# Backward-compatible aliases cho các cell phía dưới
train_df = split_dfs["train"]
unique_df = split_unique_dfs["train"]

# Sample riêng từng split để chạy nhanh nhưng vẫn cover dev/test
SAMPLE_SIZES = {"train": MAX_SAMPLES, "dev": MAX_SAMPLES, "test": MAX_SAMPLES}
split_samples = {}
for split in SPLITS:
    udf = split_unique_dfs[split]
    if SAMPLE_SIZES[split] < len(udf):
        sample_df = udf.sample(n=SAMPLE_SIZES[split], random_state=RANDOM_SEED).reset_index(drop=True)
    else:
        sample_df = udf.copy()
    sample_df["split"] = split
    sample_df["premises_nl_list"] = sample_df["premises_nl"].apply(ast.literal_eval)
    sample_df["premises_fol_list"] = sample_df["premises_fol"].apply(ast.literal_eval)
    sample_df["n_premises"] = sample_df["premises_nl_list"].apply(len)
    split_samples[split] = sample_df
    print(f"{split.title()} sampled: {len(sample_df)} records")

# Backward-compatible alias: giữ biến sample_df để các cell cũ không vỡ
sample_df = split_samples["train"]
print(f"Model: {MODEL_ID}")
print(f"Max samples per split: {MAX_SAMPLES}")
print(f"Output CSV folder: {OUTPUT_CSV.parent}")

print(f"\n--- Token Budget Analysis ---")
sys_tokens = len(tokenizer.encode(SYSTEM_PROMPT_FOL_SFT))
print(f"System prompt tokens: {sys_tokens}")
print(f"Model max_seq_length: {_MODEL_MAX_SEQ} (from config)")

for split_name in SPLITS:
    sample_df = split_samples[split_name]
    token_counts = []
    gold_fol_tokens = []
    for _, row in sample_df.iterrows():
        nl_block = format_nl_block_numbered(row["premises_nl_list"])
        user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
            {"role": "user", "content": user_msg},
        ]
        full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        n_tokens = len(tokenizer.encode(full_prompt))
        token_counts.append(n_tokens)
        gold_json = json.dumps({"premises_fol": row["premises_fol_list"]}, ensure_ascii=False)
        gold_fol_tokens.append(len(tokenizer.encode(gold_json)))

    sample_df["prompt_tokens"] = token_counts
    sample_df["gold_fol_tokens"] = gold_fol_tokens
    sample_df["total_needed"] = sample_df["prompt_tokens"] + sample_df["gold_fol_tokens"]
    sample_df["remaining"] = _MODEL_MAX_SEQ - sample_df["prompt_tokens"]

    print(f"\n[{split_name}] Prompt tokens: min={min(token_counts)}, max={max(token_counts)}, mean={sum(token_counts)/len(token_counts):.0f}")
    print(f"[{split_name}] Gold FOL tokens: min={min(gold_fol_tokens)}, max={max(gold_fol_tokens)}, mean={sum(gold_fol_tokens)/len(gold_fol_tokens):.0f}")
    actually_truncated = (sample_df["total_needed"] > _MODEL_MAX_SEQ).sum()
    print(f"[{split_name}] Samples thực sự bị cắt (prompt+gold_fol > {_MODEL_MAX_SEQ}): {actually_truncated}/{len(sample_df)} ({actually_truncated/len(sample_df):.1%})")

    top5 = sample_df.nlargest(5, "total_needed")[["record_id", "n_premises", "prompt_tokens", "gold_fol_tokens", "total_needed", "remaining"]]
    print(f"[{split_name}] Top 5 tốn token nhất:")
    for _, r in top5.iterrows():
        status = "OK" if r["total_needed"] <= _MODEL_MAX_SEQ else "TRUNCATED"
        print(f"  record_id={int(r['record_id']):4d}  premises={int(r['n_premises']):2d}  prompt={int(r['prompt_tokens'])}  gold_fol={int(r['gold_fol_tokens'])}  total={int(r['total_needed'])}  remaining={int(r['remaining'])}  [{status}]")

Train data: 647 rows, unique records=328
Dev data: 79 rows, unique records=41
Test data: 81 rows, unique records=41
Train sampled: 328 records
Dev sampled: 41 records
Test sampled: 41 records
Model: Laplaces-Red-Devils/fol-v06-cot-augmented-fol-pretrain-malls-qwen3.5-4
Max samples per split: 600
Output CSV folder: /root/Logic_Based_Educational_Queries_Project/notebooks/output

--- Token Budget Analysis ---
System prompt tokens: 1529
Model max_seq_length: 3500 (from config)



[train] Prompt tokens: min=1582, max=2372, mean=1745
[train] Gold FOL tokens: min=42, max=683, mean=189
[train] Samples thực sự bị cắt (prompt+gold_fol > 3500): 0/328 (0.0%)
[train] Top 5 tốn token nhất:
  record_id=  36  premises=34  prompt=2372  gold_fol=683  total=3055  remaining=1128  [OK]
  record_id= 408  premises=36  prompt=2364  gold_fol=533  total=2897  remaining=1136  [OK]
  record_id=  34  premises=27  prompt=2189  gold_fol=599  total=2788  remaining=1311  [OK]
  record_id= 157  premises=21  prompt=2128  gold_fol=450  total=2578  remaining=1372  [OK]
  record_id=  73  premises=22  prompt=2066  gold_fol=506  total=2572  remaining=1434  [OK]

[dev] Prompt tokens: min=1576, max=2314, mean=1744
[dev] Gold FOL tokens: min=33, max=496, mean=199
[dev] Samples thực sự bị cắt (prompt+gold_fol > 3500): 0/41 (0.0%)
[dev] Top 5 tốn token nhất:
  record_id= 409  premises=36  prompt=2314  gold_fol=496  total=2810  remaining=1186  [OK]
  record_id= 154  premises=21  prompt=2039  gold_fol=


[test] Prompt tokens: min=1607, max=2256, mean=1760
[test] Gold FOL tokens: min=71, max=502, mean=198
[test] Samples thực sự bị cắt (prompt+gold_fol > 3500): 0/41 (0.0%)
[test] Top 5 tốn token nhất:
  record_id= 407  premises=36  prompt=2256  gold_fol=502  total=2758  remaining=1244  [OK]
  record_id= 164  premises=20  prompt=2002  gold_fol=483  total=2485  remaining=1498  [OK]
  record_id= 151  premises=20  prompt=1995  gold_fol=426  total=2421  remaining=1505  [OK]
  record_id= 158  premises=20  prompt=1930  gold_fol=426  total=2356  remaining=1570  [OK]
  record_id= 100  premises=16  prompt=1903  gold_fol=367  total=2270  remaining=1597  [OK]


## 3b. Kiểm tra training data có bị cắt không

Mô phỏng đúng quá trình training: system + user + assistant → tokenize → đếm tokens.
Nếu vượt `max_seq_length=2048` → training đã cắt assistant response → model học FOL bị cụt.

In [5]:
# === Mô phỏng training: đếm tokens cho TOÀN BỘ sequence (system + user + assistant) ===
TRAIN_MAX_SEQ = 3500  # max_seq_length trong configs/fol_model.yaml khi train

# Load toàn bộ train data (không chỉ sample)
full_train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
full_train_unique = full_train_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)

truncated_records = []
all_seq_lengths = []

for _, row in full_train_unique.iterrows():
    premises_nl = ast.literal_eval(row["premises_nl"])
    premises_fol = ast.literal_eval(row["premises_fol"])
    record_id = row["record_id"]

    # Build đúng messages như training (system + user + assistant)
    nl_block = format_nl_block_numbered(premises_nl)
    user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
    assistant_msg = json.dumps({"premises_fol": premises_fol}, ensure_ascii=False)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    # Tokenize giống training (không add_generation_prompt vì đã có assistant)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokens = tokenizer.encode(full_text)
    seq_len = len(tokens)
    all_seq_lengths.append(seq_len)

    # Đếm riêng phần prompt (system + user) vs assistant
    prompt_messages = messages[:2]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    prompt_tokens = len(tokenizer.encode(prompt_text))
    assistant_tokens = seq_len - prompt_tokens

    if seq_len > TRAIN_MAX_SEQ:
        cut_amount = seq_len - TRAIN_MAX_SEQ
        truncated_records.append({
            "record_id": record_id,
            "n_premises": len(premises_nl),
            "total_tokens": seq_len,
            "prompt_tokens": prompt_tokens,
            "assistant_tokens": assistant_tokens,
            "cut_tokens": cut_amount,
            "cut_pct": cut_amount / assistant_tokens * 100 if assistant_tokens > 0 else 0,
        })

# === Report ===
print("=" * 60)
print("  TRAINING DATA TRUNCATION ANALYSIS")
print("=" * 60)
print(f"\nTotal unique records: {len(full_train_unique)}")
print(f"max_seq_length (training): {TRAIN_MAX_SEQ}")

print(f"\n--- Sequence Length Distribution ---")
print(f"  Min:  {min(all_seq_lengths)} tokens")
print(f"  Max:  {max(all_seq_lengths)} tokens")
print(f"  Mean: {sum(all_seq_lengths)/len(all_seq_lengths):.0f} tokens")
print(f"  Median: {sorted(all_seq_lengths)[len(all_seq_lengths)//2]} tokens")

n_truncated = len(truncated_records)
print(f"\n--- Truncation ---")
print(f"  Records bị cắt (>{TRAIN_MAX_SEQ}): {n_truncated}/{len(full_train_unique)} ({n_truncated/len(full_train_unique):.1%})")

if n_truncated > 0:
    trunc_df = pd.DataFrame(truncated_records)
    print(f"  Tokens bị cắt: min={trunc_df['cut_tokens'].min()}, max={trunc_df['cut_tokens'].max()}, mean={trunc_df['cut_tokens'].mean():.0f}")
    print(f"  % assistant bị cắt: mean={trunc_df['cut_pct'].mean():.1f}%, max={trunc_df['cut_pct'].max():.1f}%")

    print(f"\n--- Top 10 records bị cắt nhiều nhất ---")
    worst = trunc_df.nlargest(10, "cut_tokens")
    for _, r in worst.iterrows():
        print(f"  record_id={int(r['record_id']):4d}  premises={int(r['n_premises']):2d}  "
              f"total={int(r['total_tokens'])}tok  prompt={int(r['prompt_tokens'])}tok  "
              f"assistant={int(r['assistant_tokens'])}tok  CUT={int(r['cut_tokens'])}tok ({r['cut_pct']:.0f}%)")

    # Histogram
    print(f"\n--- Distribution: bao nhiêu tokens bị cắt ---")
    bins_labels = [(1, 50, "1-50 tok"), (51, 200, "51-200 tok"), (201, 500, "201-500 tok"), (501, 9999, "500+ tok")]
    for lo, hi, label in bins_labels:
        count = len(trunc_df[(trunc_df["cut_tokens"] >= lo) & (trunc_df["cut_tokens"] <= hi)])
        bar = "#" * count
        print(f"  {label:12s} {count:4d} records  {bar}")
else:
    print("  Không có record nào bị cắt!")

# Records an toàn (không bị cắt)
safe = len(full_train_unique) - n_truncated
print(f"\n--- Kết luận ---")
if n_truncated == 0:
    print(f"  OK: Toàn bộ {safe} records fit trong {TRAIN_MAX_SEQ} tokens. Training data không bị cắt.")
elif n_truncated / len(full_train_unique) < 0.05:
    print(f"  NHẸE: Chỉ {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Ảnh hưởng nhỏ.")
elif n_truncated / len(full_train_unique) < 0.30:
    print(f"  TRUNG BÌNH: {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Cần tăng max_seq_length khi retrain.")
else:
    print(f"  NGHIÊM TRỌNG: {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Model học từ data bị cụt!")
    print(f"  → Cần tăng max_seq_length lên ít nhất {max(all_seq_lengths)} hoặc rút ngắn system prompt.")

print("=" * 60)

  TRAINING DATA TRUNCATION ANALYSIS

Total unique records: 328
max_seq_length (training): 3500

--- Sequence Length Distribution ---
  Min:  1628 tokens
  Max:  3059 tokens
  Mean: 1939 tokens
  Median: 1900 tokens

--- Truncation ---
  Records bị cắt (>3500): 0/328 (0.0%)
  Không có record nào bị cắt!

--- Kết luận ---
  OK: Toàn bộ 328 records fit trong 3500 tokens. Training data không bị cắt.


## 4. Generate FOL for all samples

In [6]:
BATCH_SIZE = 20  # Chạy batch để tận dụng GPU

# Left-padding cho generation
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Đọc max_seq_length từ config YAML (khớp với training)
import yaml as _yaml
_cfg_path = PROJECT_ROOT / "configs" / "fol_model.yaml"
MODEL_MAX_SEQ = 3500  # default
if _cfg_path.is_file():
    _cfg = _yaml.safe_load(_cfg_path.read_text(encoding="utf-8")) or {}
    MODEL_MAX_SEQ = int((_cfg.get("model") or {}).get("max_seq_length", 3500))


def build_prompt(premises_nl: list[str]) -> str:
    """Tạo prompt cho 1 sample."""
    nl_block = format_nl_block_numbered(premises_nl)
    user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
        {"role": "user", "content": user_msg},
    ]
    # enable_thinking=False de KHOP voi training (fol_dataset.py dung enable_thinking=False).
    # Neu thieu, Qwen3 mac dinh bat thinking -> model sinh reasoning -> JSON bi cat -> parse rac.
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def generate_fol_batch(prompts: list[str]) -> tuple[list[str], int, int]:
    """Sinh FOL cho nhiều prompts cùng lúc trên GPU."""
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MODEL_MAX_SEQ,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[1]
    budget = max(MODEL_MAX_SEQ - prompt_len, 128)
    actual_max_new = min(budget, MAX_NEW_TOKENS)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=actual_max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.0,
        )

    results = []
    for row in out:
        gen_ids = row[prompt_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        results.append(text)
    return results, prompt_len, actual_max_new


def parse_fol_json(text: str) -> list[str] | None:
    """Parse JSON output -> list[str]. 3 cấp fallback."""
    if not text or not text.strip():
        return None

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if "premises_fol" in parsed and isinstance(parsed["premises_fol"], list):
                return [str(f).strip() for f in parsed["premises_fol"]]
        except json.JSONDecodeError:
            pass

    start = text.find('{"premises_fol"')
    if start == -1:
        start = text.find('"premises_fol"')
    if start >= 0:
        fol_strings = re.findall(r'"([^"]*)"', text[start:])
        values = [s for s in fol_strings if s != "premises_fol" and len(s) > 1]
        if values:
            return [v.strip() for v in values]

    lines = []
    for line in text.split("\n"):
        line = line.strip().lstrip("0123456789.)-  ")
        if any(c in line for c in "∀∃→∧∨¬↔") or re.match(r"[A-Z]\w*\(", line):
            lines.append(line)
    if lines:
        return lines

    return None


def run_split_generation(split_name: str, sample_df: pd.DataFrame) -> pd.DataFrame:
    """Sinh FOL cho một split và trả về dataframe kết quả theo premise."""
    results = []
    t_start = time.time()
    n_total = len(sample_df)
    debug_fail_count = 0

    print(f"\n=== Running split: {split_name} ({n_total} records) ===")

    for batch_start in range(0, n_total, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_total)
        batch_rows = sample_df.iloc[batch_start:batch_end]

        prompts = [build_prompt(row["premises_nl_list"]) for _, row in batch_rows.iterrows()]

        t0 = time.time()
        raw_outputs, prompt_len, actual_max_new = generate_fol_batch(prompts)
        batch_time = time.time() - t0

        for i, (_, row) in enumerate(batch_rows.iterrows()):
            premises_nl = row["premises_nl_list"]
            gold_fol = row["premises_fol_list"]
            record_id = row["record_id"]
            raw_output = raw_outputs[i]

            pred_fol = parse_fol_json(raw_output)
            json_parse_ok = pred_fol is not None
            if pred_fol is None:
                pred_fol = []

            n_gold = len(gold_fol)
            n_pred = len(pred_fol)

            for p_idx in range(max(n_gold, n_pred)):
                nl = premises_nl[p_idx] if p_idx < len(premises_nl) else "(no NL)"
                g = gold_fol[p_idx] if p_idx < n_gold else "(missing in gold)"
                p = pred_fol[p_idx] if p_idx < n_pred else "(missing in pred)"
                exact_match = g.strip() == p.strip() if p_idx < n_gold and p_idx < n_pred else False

                results.append({
                    "split": split_name,
                    "record_id": record_id,
                    "premise_idx": p_idx,
                    "premise_nl": nl,
                    "gold_fol": g,
                    "predicted_fol": p,
                    "exact_match": exact_match,
                    "json_parse_ok": json_parse_ok,
                    "n_gold": n_gold,
                    "n_pred": n_pred,
                    "count_match": n_gold == n_pred,
                    "gen_time_s": batch_time / len(batch_rows),
                    "prompt_tokens": prompt_len,
                    "max_new_tokens_used": actual_max_new,
                    "raw_output": raw_output if not json_parse_ok else "",
                })

            if not json_parse_ok and debug_fail_count < 5:
                debug_fail_count += 1
                print(f"  >>> DEBUG FAIL #{debug_fail_count} split={split_name} record_id={record_id} "
                      f"prompt={prompt_len}tok max_new={actual_max_new}tok:")
                print(f"  >>> {raw_output[:500]}")
                print()

        done = batch_end
        elapsed = time.time() - t_start
        eta = (elapsed / done) * (n_total - done) if done > 0 else 0
        batch_json_ok = sum(1 for r in raw_outputs if parse_fol_json(r) is not None)
        print(f"[{split_name} {done}/{n_total}] batch {batch_time:.1f}s ({batch_time/len(batch_rows):.1f}s/sample) "
              f"json_ok={batch_json_ok}/{len(batch_rows)} prompt={prompt_len}tok budget={actual_max_new}tok "
              f"ETA={eta/60:.1f}m")

    total_time = time.time() - t_start
    print(f"Done {split_name}! {n_total} records, {len(results)} premises, {total_time/60:.1f} minutes")
    print(f"Speed {split_name}: {total_time/n_total:.1f}s/sample (batch={BATCH_SIZE})")
    return pd.DataFrame(results)


# === Generate FOL for train/dev/test ===
all_results_df = []
for split_name in SPLITS:
    split_df = split_samples[split_name]
    split_results = run_split_generation(split_name, split_df)
    all_results_df.append(split_results)
    split_csv = OUTPUT_CSV.with_name(f"{OUTPUT_CSV.stem}_{split_name}{OUTPUT_CSV.suffix}")
    split_results.to_csv(split_csv, index=False, encoding="utf-8-sig")
    print(f"Saved split CSV: {split_csv}")

df_results = pd.concat(all_results_df, ignore_index=True)
df_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved combined CSV: {OUTPUT_CSV}")
print(f"Total rows: {len(df_results)}")
print(f"\nQuick stats:")
print(f"  JSON parse OK:    {df_results['json_parse_ok'].mean():.1%}")
print(f"  Count match:      {df_results.groupby(['split', 'record_id'])['count_match'].first().mean():.1%}")
print(f"  Exact match:      {df_results['exact_match'].mean():.1%}")
df_results.head(10)


=== Running split: train (328 records) ===


/usr/local/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[train 20/328] batch 133.7s (6.7s/sample) json_ok=20/20 prompt=1788tok budget=650tok ETA=34.3m


[train 40/328] batch 245.2s (12.3s/sample) json_ok=20/20 prompt=2374tok budget=650tok ETA=45.5m


[train 60/328] batch 120.6s (6.0s/sample) json_ok=20/20 prompt=1910tok budget=650tok ETA=37.2m


...

[dev 41/41] batch 123.6s (123.6s/sample) json_ok=1/1 prompt=2316tok budget=650tok ETA=0.0m
Done dev! 41 records, 450 premises, 7.3 minutes
Speed dev: 10.6s/sample (batch=20)
Saved split CSV: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic_dev.csv

=== Running split: test (41 records) ===


[test 20/41] batch 177.1s (8.9s/sample) json_ok=20/20 prompt=2004tok budget=650tok ETA=3.1m


[test 40/41] batch 109.1s (5.5s/sample) json_ok=20/20 prompt=1913tok budget=650tok ETA=0.1m


[test 41/41] batch 93.7s (93.7s/sample) json_ok=1/1 prompt=2258tok budget=650tok ETA=0.0m
Done test! 41 records, 470 premises, 6.3 minutes
Speed test: 9.3s/sample (batch=20)
Saved split CSV: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic_test.csv

Saved combined CSV: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic.csv
Total rows: 4460

Quick stats:
  JSON parse OK:    100.0%
  Count match:      96.8%
  Exact match:      81.7%


,split,record_id,premise_idx,premise_nl,gold_fol,predicted_fol,exact_match,json_parse_ok,n_gold,n_pred,count_match,gen_time_s,prompt_tokens,max_new_tokens_used,raw_output
0,train,0,0,"If a Python code is well-tested, then the proj...",∀x (WT(x) → O(x)),∀x (WT(x) → O(x)),True,True,14,14,True,6.684914,1788,650,
1,train,0,1,If a Python code does not follow PEP 8 standar...,∀x (¬PEP8(x) → ¬WT(x)),∀x (¬PEP8(x) → ¬WT(x)),True,True,14,14,True,6.684914,1788,650,
2,train,0,2,All Python projects are easy to maintain.,∀x (EM(x)),∀x (EM(x)),True,True,14,14,True,6.684914,1788,650,
3,train,0,3,All Python code is well-tested.,∀x (WT(x)),∀x (WT(x)),True,True,14,14,True,6.684914,1788,650,
4,train,0,4,"If a Python code follows PEP 8 standards, then...",∀x (PEP8(x) → EM(x)),∀x (PEP8(x) → EM(x)),True,True,14,14,True,6.684914,1788,650,
5,train,0,5,"If a Python code is well-tested, then it follo...",∀x (WT(x) → PEP8(x)),∀x (WT(x) → PEP8(x)),True,True,14,14,True,6.684914,1788,650,
6,train,0,6,"If a Python project is well-structured, then i...",∀x (WS(x) → O(x)),∀x (WS(x) → O(x)),True,True,14,14,True,6.684914,1788,650,
7,train,0,7,"If a Python project is easy to maintain, then ...",∀x (EM(x) → WT(x)),∀x (EM(x) → WT(x)),True,True,14,14,True,6.684914,1788,650,
8,train,0,8,"If a Python project is optimized, then it has ...",∀x (O(x) → CR(x)),∀x (O(x) → CR(x)),True,True,14,14,True,6.684914,1788,650,
9,train,0,9,All Python projects are well-structured.,∀x (WS(x)),∀x (WS(x)),True,True,14,14,True,6.684914,1788,650,


### Sanity check: enable_thinking True vs False
Xac nhan bug. Ky vong: `False` ra JSON sach `{"premises_fol": [...]}`, `True` ra reasoning prose.


In [7]:
# So sanh 1 sample voi enable_thinking True vs False
_row = sample_df.iloc[0]
_msgs = [
    {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
    {"role": "user", "content": USER_TEMPLATE_FOL_SFT.format(
        premises_nl=format_nl_block_numbered(_row["premises_nl_list"])).strip()},
]
for _th in (True, False):
    _p = tokenizer.apply_chat_template(
        _msgs, tokenize=False, add_generation_prompt=True, enable_thinking=_th
    )
    _ids = tokenizer(_p, return_tensors="pt").to(model.device)
    _plen = _ids["input_ids"].shape[1]
    with torch.no_grad():
        _out = model.generate(
            **_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    _txt = tokenizer.decode(_out[0][_plen:], skip_special_tokens=True).strip()
    print("")
    print(f"===== enable_thinking={_th} =====")
    print(_txt[:600])


/usr/local/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



===== enable_thinking=True =====
Thinking Process:

1.  **Analyze the Request:**
    *   Input: 7 natural language premises.
    *   Task: Convert each premise into a precise First-Order Logic (FOL) formula.
    *   Output Format: JSON object `{"premises_fol": ["...", "..."]}`.
    *   Constraints: No markdown, no explanation, single lowercase variables, consistent predicate naming, logical structure accuracy.

2.  **Analyze Each Premise:**

    *   **Premise 1:** "Lecturers with a Master's degree can teach undergraduate courses."
        *   Subject: Lecturers (∀x).
        *   Condition: Has Master's (M(x)).
        *   Res



===== enable_thinking=False =====
{"premises_fol": ["∀x (Lecturer(x) ∧ (Masters(x) ∨ HigherDegree(x)) → TeachUndergrad(x))", "∀x (Lecturer(x) ∧ HigherDegree(x) → TeachUndergrad(x))", "∀x (PhD(x) → HigherThanMasters(x))", "∀x (Masters(x) → HigherThanBachlers(x))", "∀x (∀y (HigherThan(y, x) ∧ HigherThan(x, z) → HigherThan(y, z)))", "∀x (DeptHead(x) → HigherThanBachlers(x))", "DeptHead(John) ∧ PhD(John)"]}


In [8]:
# === Generate FOL — BATCHED ===
results = []
t_start = time.time()
n_total = len(sample_df)
debug_fail_count = 0

for batch_start in range(0, n_total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n_total)
    batch_rows = sample_df.iloc[batch_start:batch_end]

    # Build prompts
    prompts = [build_prompt(row["premises_nl_list"]) for _, row in batch_rows.iterrows()]

    # Generate batch
    t0 = time.time()
    raw_outputs, prompt_len, actual_max_new = generate_fol_batch(prompts)
    batch_time = time.time() - t0

    # Process từng sample
    for i, (_, row) in enumerate(batch_rows.iterrows()):
        premises_nl = row["premises_nl_list"]
        gold_fol = row["premises_fol_list"]
        record_id = row["record_id"]
        raw_output = raw_outputs[i]

        pred_fol = parse_fol_json(raw_output)
        json_parse_ok = pred_fol is not None
        if pred_fol is None:
            pred_fol = []

        n_gold = len(gold_fol)
        n_pred = len(pred_fol)

        for p_idx in range(max(n_gold, n_pred)):
            nl = premises_nl[p_idx] if p_idx < len(premises_nl) else "(no NL)"
            g = gold_fol[p_idx] if p_idx < n_gold else "(missing in gold)"
            p = pred_fol[p_idx] if p_idx < n_pred else "(missing in pred)"
            exact_match = g.strip() == p.strip() if p_idx < n_gold and p_idx < n_pred else False

            results.append({
                "record_id": record_id,
                "premise_idx": p_idx,
                "premise_nl": nl,
                "gold_fol": g,
                "predicted_fol": p,
                "exact_match": exact_match,
                "json_parse_ok": json_parse_ok,
                "n_gold": n_gold,
                "n_pred": n_pred,
                "count_match": n_gold == n_pred,
                "gen_time_s": batch_time / len(batch_rows),
                "prompt_tokens": prompt_len,
                "max_new_tokens_used": actual_max_new,
                "raw_output": raw_output if not json_parse_ok else "",
            })

        # Debug fail
        if not json_parse_ok and debug_fail_count < 5:
            debug_fail_count += 1
            print(f"  >>> DEBUG FAIL #{debug_fail_count} record_id={record_id} "
                  f"prompt={prompt_len}tok max_new={actual_max_new}tok:")
            print(f"  >>> {raw_output[:500]}")
            print()

    # Progress
    done = batch_end
    elapsed = time.time() - t_start
    eta = (elapsed / done) * (n_total - done) if done > 0 else 0
    batch_json_ok = sum(1 for r in raw_outputs if parse_fol_json(r) is not None)
    print(f"[{done}/{n_total}] batch {batch_time:.1f}s ({batch_time/len(batch_rows):.1f}s/sample) "
          f"json_ok={batch_json_ok}/{len(batch_rows)} prompt={prompt_len}tok budget={actual_max_new}tok "
          f"ETA={eta/60:.1f}m")

total_time = time.time() - t_start
print(f"\nDone! {n_total} records, {len(results)} premises, {total_time/60:.1f} minutes")
print(f"Speed: {total_time/n_total:.1f}s/sample (batch={BATCH_SIZE})")

[20/41] batch 177.3s (8.9s/sample) json_ok=20/20 prompt=2004tok budget=650tok ETA=3.1m


[40/41] batch 109.9s (5.5s/sample) json_ok=20/20 prompt=1913tok budget=650tok ETA=0.1m


[41/41] batch 92.3s (92.3s/sample) json_ok=1/1 prompt=2258tok budget=650tok ETA=0.0m

Done! 41 records, 470 premises, 6.3 minutes
Speed: 9.3s/sample (batch=20)


## 5. Export CSV

In [9]:
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")
print(f"Total rows: {len(df_results)}")
print(f"\nQuick stats:")
print(f"  JSON parse OK:    {df_results['json_parse_ok'].mean():.1%}")
print(f"  Count match:      {df_results.groupby('record_id')['count_match'].first().mean():.1%}")
print(f"  Exact match:      {df_results['exact_match'].mean():.1%}")
df_results.head(10)

Saved: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic.csv
Total rows: 470

Quick stats:
  JSON parse OK:    100.0%
  Count match:      92.7%
  Exact match:      30.2%


,record_id,premise_idx,premise_nl,gold_fol,predicted_fol,exact_match,json_parse_ok,n_gold,n_pred,count_match,gen_time_s,prompt_tokens,max_new_tokens_used,raw_output
0,14,0,Lecturers with a Master's degree can teach und...,"∀x (has_degree(x, MSc) → teach_undergrad(x))",∀x (Lecturer(x) ∧ (Masters(x) ∨ HigherDegree(x...,False,True,7,7,True,8.866289,2004,650,
1,14,1,Lecturers with a degree higher than a Master's...,"∀x (∀d ((higher(d, MSc) ∧ has_degree(x, d)) → ...",∀x (∀d ((HigherDegree(d) → TeachUndergrad(x)) ...,False,True,7,7,True,8.866289,2004,650,
2,14,2,A PhD is higher than a Master's degree.,"higher(PhD, MSc)","∀x (PhD(x) → HigherThan(x, M))",False,True,7,7,True,8.866289,2004,650,
3,14,3,A Master's degree is higher than a Bachelor's ...,"higher(MSc, BSc)","∀x (Masters(x) → HigherThan(x, Bachelors))",False,True,7,7,True,8.866289,2004,650,
4,14,4,"If degree A is higher than degree B, and degre...","∀a (∀b (∀c ((higher(a, b) ∧ higher(b, c)) → hi...","∀x (∀y (∀z ((HigherThan(x, y) ∧ HigherThan(y, ...",False,True,7,7,True,8.866289,2004,650,
5,14,5,Department heads must hold a degree higher tha...,"∀x (department_head(x) → (∃d, has_degree(x, d)...",∀x (DeptHead(x) → HigherDegree(x)),False,True,7,7,True,8.866289,2004,650,
6,14,6,Dr. John is a department head with a PhD.,"department_head(John) ∧ has_degree(John, PhD)",HigherDegree(PhD),False,True,7,7,True,8.866289,2004,650,
7,19,0,"If a person has a research background, then th...",∀x (ResearchBackground(x) → StrongQualificatio...,∀x (R(x) → Q(x)),False,True,6,6,True,8.866289,2004,650,
8,19,1,"If a person has strong qualifications, then th...",∀x (StrongQualifications(x) → SeniorRoleSuitab...,∀x (Q(x) → S(x)),False,True,6,6,True,8.866289,2004,650,
9,19,2,Every person is enrolled in the company’s deve...,∀x (DevelopmentProgram(x)),∀x E(x),False,True,6,6,True,8.866289,2004,650,


## 6. Z3 Parse Check

Dùng parser từ `src/evaluation/fol_parser.py` để check từng FOL formula có parse được không.

In [10]:
from evaluation.fol_parser import parse_fol, FOLParseError
from evaluation.fol_z3_translator import safe_fol_string_to_z3

def check_z3_parse(fol_str: str) -> tuple[bool, str]:
    """Check if a FOL string can be parsed and translated to Z3.
    Returns (success: bool, error_msg: str).
    """
    if not fol_str or fol_str.startswith("(missing"):
        return False, "empty_or_missing"
    
    # Step 1: Parse to AST
    try:
        ast_node = parse_fol(fol_str)
    except FOLParseError as e:
        return False, f"parse_error: {e}"
    except Exception as e:
        return False, f"unexpected_error: {e}"
    
    # Step 2: Translate AST to Z3
    cache = {}
    z3_expr = safe_fol_string_to_z3(fol_str, cache)
    if z3_expr is None:
        return False, "z3_translation_failed"
    
    return True, ""


# Run Z3 check on all predicted FOL
print("Running Z3 parse check on predicted FOL...")
z3_results_pred = []
for _, row in df_results.iterrows():
    ok, err = check_z3_parse(row["predicted_fol"])
    z3_results_pred.append({"z3_parse_ok": ok, "z3_error_msg": err})

z3_pred_df = pd.DataFrame(z3_results_pred)
df_results["z3_parse_ok"] = z3_pred_df["z3_parse_ok"]
df_results["z3_error_msg"] = z3_pred_df["z3_error_msg"]

# Also check gold FOL (to see if gold itself has issues)
print("Running Z3 parse check on gold FOL...")
z3_results_gold = []
for _, row in df_results.iterrows():
    ok, err = check_z3_parse(row["gold_fol"])
    z3_results_gold.append({"gold_z3_parse_ok": ok, "gold_z3_error_msg": err})

z3_gold_df = pd.DataFrame(z3_results_gold)
df_results["gold_z3_parse_ok"] = z3_gold_df["gold_z3_parse_ok"]
df_results["gold_z3_error_msg"] = z3_gold_df["gold_z3_error_msg"]

# Save updated CSV
df_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\nUpdated CSV saved: {OUTPUT_CSV}")

Running Z3 parse check on predicted FOL...


Running Z3 parse check on gold FOL...



Updated CSV saved: /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic.csv


## 7. Diagnostic Report

In [11]:
total = len(df_results)
pred_not_missing = df_results[~df_results["predicted_fol"].str.startswith("(missing")]
total_valid = len(pred_not_missing)

print("=" * 60)
print("  FOL DIAGNOSTIC REPORT")
print("=" * 60)

# --- Overall stats ---
print(f"\n--- Overall ---")
print(f"Total premises:          {total}")
print(f"Valid predictions:       {total_valid} ({total_valid/total:.1%})")
print(f"Missing predictions:     {total - total_valid}")

n_records = df_results["record_id"].nunique()
json_ok_records = df_results.groupby("record_id")["json_parse_ok"].first()
print(f"\n--- JSON Parse (model output -> list) ---")
print(f"Records with valid JSON: {json_ok_records.sum()}/{n_records} ({json_ok_records.mean():.1%})")

count_match = df_results.groupby("record_id")["count_match"].first()
print(f"\n--- Count Match (n_gold == n_pred) ---")
print(f"Records with count match: {count_match.sum()}/{n_records} ({count_match.mean():.1%})")

# --- Z3 Parse ---
print(f"\n--- Z3 Parse (predicted FOL -> Z3) ---")
z3_ok = pred_not_missing["z3_parse_ok"].sum()
print(f"Z3 parse OK:  {z3_ok}/{total_valid} ({z3_ok/total_valid:.1%})")
print(f"Z3 parse FAIL: {total_valid - z3_ok}/{total_valid} ({(total_valid-z3_ok)/total_valid:.1%})")

print(f"\n--- Z3 Parse (gold FOL -> Z3) ---")
gold_valid = df_results[~df_results["gold_fol"].str.startswith("(missing")]
gold_z3_ok = gold_valid["gold_z3_parse_ok"].sum()
print(f"Gold Z3 parse OK:  {gold_z3_ok}/{len(gold_valid)} ({gold_z3_ok/len(gold_valid):.1%})")

# --- Exact Match ---
print(f"\n--- Exact Match (pred == gold string) ---")
em = df_results["exact_match"].sum()
print(f"Exact match: {em}/{total} ({em/total:.1%})")

print(f"\n" + "=" * 60)

  FOL DIAGNOSTIC REPORT

--- Overall ---
Total premises:          470
Valid predictions:       458 (97.4%)
Missing predictions:     12

--- JSON Parse (model output -> list) ---
Records with valid JSON: 41/41 (100.0%)

--- Count Match (n_gold == n_pred) ---
Records with count match: 38/41 (92.7%)

--- Z3 Parse (predicted FOL -> Z3) ---
Z3 parse OK:  457/458 (99.8%)
Z3 parse FAIL: 1/458 (0.2%)

--- Z3 Parse (gold FOL -> Z3) ---
Gold Z3 parse OK:  468/470 (99.6%)

--- Exact Match (pred == gold string) ---
Exact match: 142/470 (30.2%)



In [12]:
# --- Z3 Parse Error Analysis ---
print("\n--- Top Z3 Parse Error Patterns (predicted FOL) ---\n")

failed = pred_not_missing[~pred_not_missing["z3_parse_ok"]].copy()

if len(failed) > 0:
    # Categorize errors
    def categorize_error(err_msg: str, fol: str) -> str:
        if "parse_error" in err_msg:
            if "Extra tokens" in err_msg:
                return "extra_tokens (unbalanced parens or trailing junk)"
            if "Expected variable" in err_msg:
                return "bad_variable (multi-char var after quantifier)"
            if "Unexpected token" in err_msg:
                return "unexpected_token"
            if "Unexpected end" in err_msg:
                return "unexpected_eof"
            return "other_parse_error"
        if "z3_translation" in err_msg:
            return "z3_translation_failed"
        if "empty" in err_msg:
            return "empty"
        return "unknown"

    failed["error_category"] = failed.apply(
        lambda r: categorize_error(r["z3_error_msg"], r["predicted_fol"]), axis=1
    )

    # Count by category
    cat_counts = failed["error_category"].value_counts()
    for cat, count in cat_counts.items():
        pct = count / total_valid * 100
        print(f"  {cat:50s} {count:5d} ({pct:.1f}%)")

    print(f"\n--- Sample failures per category ---\n")
    for cat in cat_counts.index[:5]:  # Top 5 categories
        samples = failed[failed["error_category"] == cat].head(3)
        print(f"\n  [{cat}]")
        for _, s in samples.iterrows():
            print(f"    FOL:   {s['predicted_fol'][:100]}")
            print(f"    Error: {s['z3_error_msg'][:100]}")
            print()
else:
    print("  No Z3 parse failures!")


--- Top Z3 Parse Error Patterns (predicted FOL) ---

  extra_tokens (unbalanced parens or trailing junk)      1 (0.2%)

--- Sample failures per category ---


  [extra_tokens (unbalanced parens or trailing junk)]
    FOL:   J ∈ S
    Error: parse_error: Extra tokens after complete formula at pos 1: ['S']



In [13]:
# --- Gold FOL parse failures (if any) ---
print("\n--- Gold FOL that ALSO fail Z3 parse ---\n")

gold_failed = gold_valid[~gold_valid["gold_z3_parse_ok"]]
if len(gold_failed) > 0:
    print(f"Gold Z3 failures: {len(gold_failed)}/{len(gold_valid)} ({len(gold_failed)/len(gold_valid):.1%})")
    print()
    for _, s in gold_failed.head(10).iterrows():
        print(f"  record_id={s['record_id']} premise_idx={s['premise_idx']}")
        print(f"  Gold FOL: {s['gold_fol'][:120]}")
        print(f"  Error:    {s['gold_z3_error_msg'][:120]}")
        print()
else:
    print("  All gold FOL parsed successfully by Z3.")


--- Gold FOL that ALSO fail Z3 parse ---

Gold Z3 failures: 2/470 (0.4%)

  record_id=14 premise_idx=5
  Gold FOL: ∀x (department_head(x) → (∃d, has_degree(x, d) ∧ higher(d, BSc)))
  Error:    parse_error: Expected ')' at pos 15, got ','

  record_id=134 premise_idx=1
  Gold FOL: ∀x (Student(x) ∧ ≥(Score(x, Calculus1), 4) → Took(x, Calculus2))
  Error:    parse_error: Expected ')' at pos 16, got ','



In [14]:
# --- Per-record summary: how many premises parse vs fail ---
print("\n--- Per-record Z3 parse rate ---\n")

record_stats = pred_not_missing.groupby("record_id").agg(
    total_premises=("z3_parse_ok", "count"),
    z3_ok=("z3_parse_ok", "sum"),
    exact_matches=("exact_match", "sum"),
).reset_index()
record_stats["z3_fail"] = record_stats["total_premises"] - record_stats["z3_ok"]
record_stats["z3_rate"] = record_stats["z3_ok"] / record_stats["total_premises"]

# Distribution
print("Z3 parse rate distribution:")
bins = [0, 0.5, 0.8, 0.99, 1.0]
labels = ["< 50%", "50-80%", "80-99%", "100%"]
record_stats["rate_bin"] = pd.cut(record_stats["z3_rate"], bins=bins, labels=labels, include_lowest=True)
dist = record_stats["rate_bin"].value_counts().sort_index()
for label, count in dist.items():
    pct = count / len(record_stats) * 100
    bar = '#' * int(pct / 2)
    print(f"  {label:10s} {count:4d} records ({pct:5.1f}%) {bar}")

print(f"\nWorst records (most Z3 failures):")
worst = record_stats.nlargest(10, "z3_fail")
for _, r in worst.iterrows():
    print(f"  record_id={int(r['record_id']):4d}  "
          f"z3_ok={int(r['z3_ok'])}/{int(r['total_premises'])}  "
          f"z3_fail={int(r['z3_fail'])}  "
          f"exact_match={int(r['exact_matches'])}")


--- Per-record Z3 parse rate ---

Z3 parse rate distribution:
  < 50%         0 records (  0.0%) 
  50-80%        1 records (  2.4%) #
  80-99%        0 records (  0.0%) 
  100%         40 records ( 97.6%) ################################################

Worst records (most Z3 failures):
  record_id= 135  z3_ok=4/5  z3_fail=1  exact_match=0
  record_id=  14  z3_ok=7/7  z3_fail=0  exact_match=0
  record_id=  19  z3_ok=6/6  z3_fail=0  exact_match=0
  record_id=  23  z3_ok=14/14  z3_fail=0  exact_match=0
  record_id=  54  z3_ok=12/12  z3_fail=0  exact_match=0
  record_id=  57  z3_ok=12/12  z3_fail=0  exact_match=0
  record_id= 100  z3_ok=16/16  z3_fail=0  exact_match=2
  record_id= 123  z3_ok=6/6  z3_fail=0  exact_match=0
  record_id= 134  z3_ok=3/3  z3_fail=0  exact_match=0
  record_id= 151  z3_ok=20/20  z3_fail=0  exact_match=2


## 8. Predicate Collision Check

Kiểm tra xem predicted FOL có bị trùng predicate name cho 2 concepts khác nhau không.

In [15]:
import re

def extract_predicates(fol_str: str) -> set[str]:
    """Extract predicate names from FOL string."""
    # Match: Name( or Name(x
    return set(re.findall(r'([A-Za-z_][A-Za-z0-9_]*)\s*\(', fol_str))


print("--- Predicate Analysis per Record ---\n")

pred_analysis = []
for record_id, group in pred_not_missing.groupby("record_id"):
    all_preds = set()
    for fol in group["predicted_fol"]:
        all_preds.update(extract_predicates(fol))

    gold_preds = set()
    for fol in group["gold_fol"]:
        gold_preds.update(extract_predicates(fol))

    pred_analysis.append({
        "record_id": record_id,
        "n_pred_predicates": len(all_preds),
        "n_gold_predicates": len(gold_preds),
        "pred_predicates": all_preds,
        "gold_predicates": gold_preds,
    })

pred_df = pd.DataFrame(pred_analysis)
pred_df["pred_vs_gold_ratio"] = pred_df["n_pred_predicates"] / pred_df["n_gold_predicates"].clip(lower=1)

# Records where predicted has far fewer predicates than gold (possible collision)
collision_suspects = pred_df[pred_df["pred_vs_gold_ratio"] < 0.6]
print(f"Collision suspects (pred predicates < 60% of gold): {len(collision_suspects)}/{len(pred_df)}")
if len(collision_suspects) > 0:
    for _, r in collision_suspects.head(5).iterrows():
        print(f"\n  record_id={int(r['record_id'])}:")
        print(f"    Gold predicates ({r['n_gold_predicates']}): {r['gold_predicates']}")
        print(f"    Pred predicates ({r['n_pred_predicates']}): {r['pred_predicates']}")

# Records where predicted has far more predicates (no reuse)
no_reuse = pred_df[pred_df["pred_vs_gold_ratio"] > 2.0]
print(f"\nNo-reuse suspects (pred predicates > 200% of gold): {len(no_reuse)}/{len(pred_df)}")
if len(no_reuse) > 0:
    for _, r in no_reuse.head(5).iterrows():
        print(f"\n  record_id={int(r['record_id'])}:")
        print(f"    Gold predicates ({r['n_gold_predicates']}): {r['gold_predicates']}")
        print(f"    Pred predicates ({r['n_pred_predicates']}): {r['pred_predicates']}")

--- Predicate Analysis per Record ---

Collision suspects (pred predicates < 60% of gold): 0/41

No-reuse suspects (pred predicates > 200% of gold): 2/41

  record_id=329:
    Gold predicates (7): {'x', 'S', 'Q', 'U', 'R', 'T', 'P'}
    Pred predicates (22): {'J', 'S', 'A', 'K', 'Q', 'U', 'E', 'M', 'Y', 'L', 'B', 'F', 'W', 'T', 'C', 'x', 'G', 'N', 'D', 'R', 'O', 'P'}

  record_id=330:
    Gold predicates (7): {'x', 'S', 'Q', 'U', 'R', 'T', 'P'}
    Pred predicates (22): {'J', 'S', 'A', 'K', 'Q', 'U', 'E', 'M', 'Y', 'L', 'B', 'F', 'W', 'T', 'C', 'x', 'G', 'N', 'D', 'R', 'O', 'P'}


## 9. Summary

In [16]:
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"""  
  Model:            {MODEL_ID}
  Samples:           {len(sample_df)} records, {total} premises
  
  JSON parse rate:   {json_ok_records.mean():.1%}
  Count match rate:  {count_match.mean():.1%}
  Exact match rate:  {em/total:.1%}
  
  Z3 parse (pred):   {z3_ok}/{total_valid} ({z3_ok/total_valid:.1%})
  Z3 parse (gold):   {gold_z3_ok}/{len(gold_valid)} ({gold_z3_ok/len(gold_valid):.1%})
  
  Collision suspects: {len(collision_suspects)}
  No-reuse suspects:  {len(no_reuse)}
  
  Output CSV:        {OUTPUT_CSV}
""")
print("=" * 60)

  FINAL SUMMARY
  
  Model:            Laplaces-Red-Devils/fol-v06-cot-augmented-fol-pretrain-malls-qwen3.5-4
  Samples:           41 records, 470 premises

  JSON parse rate:   100.0%
  Count match rate:  92.7%
  Exact match rate:  30.2%

  Z3 parse (pred):   457/458 (99.8%)
  Z3 parse (gold):   468/470 (99.6%)

  Collision suspects: 0
  No-reuse suspects:  2

  Output CSV:        /root/Logic_Based_Educational_Queries_Project/notebooks/output/fol_diagnostic.csv

